In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")

BASE_DIR = Path("..").resolve()
sys.path.insert(0, str(BASE_DIR / "scripts"))

from checks import bootstrap_mean_test, overlap_coefficient

In [ ]:
# Load data
data_path = BASE_DIR / "aggregated_results" / "aggregated_results.csv"

if not data_path.exists():
    raise FileNotFoundError(
        f"Missing {data_path.name}. Run scripts/aggregate_conclusions.py first."
    )

df = pd.read_csv(data_path, keep_default_na=False)

In [ ]:
# # make one subset with the results from distribution diff-null
# # and one subset with the results from alt
# df_diff_null = df[df["distribution"] == "diff-null"].copy()
# df_alt = df[df["distribution"] == "alt"].copy()

# # get the datasets that have diff-null results and only keep those in the alt dataset
# datasets_with_diff_null = df_diff_null["dataset"].unique()
# df_alt = df_alt[df_alt["dataset"].isin(datasets_with_diff_null)].copy()

In [ ]:
# # Bootstrap mean test + overlap coefficient per dataset (alt vs diff-null)
# rng = np.random.default_rng(42)
# OVL_THRESHOLD = 0.2
# MEAN_ALPHA = 0.05

# datasets = sorted(datasets_with_diff_null)

# results = []
# for ds in datasets:
#     alt_scores  = df_alt.loc[df_alt["dataset"] == ds, "response"].values.astype(float)
#     null_scores = df_diff_null.loc[df_diff_null["dataset"] == ds, "response"].values.astype(float)

#     if len(alt_scores) == 0 or len(null_scores) == 0:
#         continue

#     obs_mean, p_value, _, ci_95 = bootstrap_mean_test(alt_scores, mu0=50.0, rng=rng)
#     ovl = overlap_coefficient(alt_scores, null_scores)

#     results.append({
#         "dataset":  ds,
#         "n_alt":    len(alt_scores),
#         "n_null":   len(null_scores),
#         "obs_mean": obs_mean,
#         "p_value":  p_value,
#         "ci_lo":    ci_95[0],
#         "ci_hi":    ci_95[1],
#         "ovl":      ovl,
#     })

# results_df = pd.DataFrame(results)
# results_df["mean_sig"] = results_df["p_value"] < MEAN_ALPHA
# results_df["low_ovl"]  = results_df["ovl"] < OVL_THRESHOLD

# print(results_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# # Scatterplot: bootstrap mean vs overlap coefficient (alt vs diff-null)
# from adjustText import adjust_text

# categories = ["Confident Yes", "Low Signal", "Analysis Failure", "No Signal"]
# cat_colors = {
#     "Confident Yes":    "seagreen",
#     "Low Signal":       "goldenrod",
#     "Analysis Failure": "mediumpurple",
#     "No Signal":        "lightcoral",
# }

# LABEL_FS = 13
# TICK_FS  = 11
# ANNOT_FS = 9
# TITLE_FS = 13

# def classify(row):
#     if row["mean_sig"] and row["low_ovl"]:
#         return "Confident Yes"
#     elif row["mean_sig"] and not row["low_ovl"]:
#         return "Analysis Failure"
#     elif not row["mean_sig"] and row["low_ovl"]:
#         return "Low Signal"
#     else:
#         return "No Signal"

# results_df["category"] = results_df.apply(classify, axis=1)

# fig, ax = plt.subplots(figsize=(8, 6))

# texts = []
# for cat in categories:
#     sub = results_df[results_df["category"] == cat]
#     ax.scatter(sub["ovl"], sub["obs_mean"],
#                color=cat_colors[cat], s=90, label=cat,
#                edgecolor="white", linewidth=0.6, zorder=3)
#     for _, row in sub.iterrows():
#         texts.append(ax.text(
#             row["ovl"], row["obs_mean"], row["dataset"],
#             fontsize=ANNOT_FS, fontfamily="DejaVu Sans Mono",
#         ))

# adjust_text(
#     texts, ax=ax,
#     expand=(1.3, 1.5),
#     arrowprops=dict(arrowstyle="-", color="gray", lw=0.6),
# )

# ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, zorder=1)
# ax.axvline(OVL_THRESHOLD, color="gray", linestyle="--", linewidth=0.8, zorder=1)
# ax.set_xlabel("Overlap coefficient (Scott's rule)", fontsize=LABEL_FS)
# ax.set_ylabel("Alt response mean", fontsize=LABEL_FS)
# ax.tick_params(labelsize=TICK_FS)
# ax.set_xlim(-0.05, 1.05)
# ax.set_title(
#     f"Bootstrap mean test \u00d7 overlap  (threshold = {OVL_THRESHOLD},  \u03b1 = {MEAN_ALPHA})\n"
#     "alt vs diff-null",
#     fontsize=TITLE_FS,
# )
# ax.legend(loc="upper right", fontsize=ANNOT_FS)
# sns.despine()
# plt.tight_layout()
# plt.show()

In [ ]:
# # Ridgeline (ggridges-style) density plot: alt vs diff-null per dataset
# from scipy.stats import gaussian_kde
# from matplotlib.patches import Patch

# X_GRID     = np.linspace(0, 100, 500)
# ROW_HEIGHT = 1.0
# RIDGE_SCALE = 0.75
# NULL_COLOR  = "steelblue"
# ALT_COLOR   = "tomato"

# n_ds = len(datasets)
# fig, ax = plt.subplots(figsize=(10, n_ds * 0.7 + 1.5))

# for i, ds in enumerate(reversed(datasets)):
#     baseline    = i * ROW_HEIGHT
#     alt_scores  = df_alt.loc[df_alt["dataset"] == ds, "response"].values.astype(float)
#     null_scores = df_diff_null.loc[df_diff_null["dataset"] == ds, "response"].values.astype(float)

#     ax.axhline(baseline, color="gray", linewidth=0.4, zorder=0)

#     for scores, color in [(null_scores, NULL_COLOR), (alt_scores, ALT_COLOR)]:
#         if len(scores) < 2:
#             continue
#         kde  = gaussian_kde(scores)
#         dens = kde(X_GRID)
#         dens = dens / dens.max() * RIDGE_SCALE
#         ax.fill_between(X_GRID, baseline, baseline + dens,
#                         color=color, alpha=0.45, linewidth=0)
#         ax.plot(X_GRID, baseline + dens, color=color, linewidth=1.2)

#     ax.text(-1, baseline + RIDGE_SCALE * 0.1, ds,
#             ha="right", va="bottom", fontsize=9)

# ax.legend(
#     handles=[Patch(color=NULL_COLOR, alpha=0.7, label="diff-null"),
#              Patch(color=ALT_COLOR,  alpha=0.7, label="alt")],
#     loc="upper right", fontsize=9,
# )
# ax.set_xlim(0, 100)
# ax.set_ylim(-ROW_HEIGHT * 0.3, n_ds * ROW_HEIGHT + 0.2)
# ax.set_xlabel("Response score", fontsize=11)
# ax.set_yticks([])
# ax.set_title("diff-null vs alt response distributions", fontsize=12)
# sns.despine(left=True)
# plt.tight_layout()
# plt.show()

# What if PVE = 0.1 is used as the null?

Same analysis as above, but instead of using the diff-null distribution as the reference,
we use runs generated at PVE = 0.01 as the null group and compare them against the alt responses.

In [ ]:
from adjustText import adjust_text

categories = ["Confident Yes", "Low Signal", "Analysis Failure", "No Signal"]
cat_colors = {
    "Confident Yes":    "seagreen",
    "Low Signal":       "goldenrod",
    "Analysis Failure": "mediumpurple",
    "No Signal":        "lightcoral",
}

def classify(row):
    if row["alt_mean_sig"] and row["low_ovl"]:
        return "Confident Yes"
    elif row["alt_mean_sig"] and not row["low_ovl"]:
        if row["null_obs_mean"] > 50.0:
            return "Low Signal"
        return "Analysis Failure"
    elif not row["alt_mean_sig"] and row["low_ovl"]:
        return "Low Signal"
    else:
        return "No Signal"

In [ ]:
# Load PVE data and build pve=0.1 null group
pve_path = BASE_DIR / "aggregated_results" / "aggregated_pve_results.csv"
df_pve = pd.read_csv(pve_path, keep_default_na=False)

df = pd.read_csv(data_path, keep_default_na=False)
df_alt = df[df["distribution"] == "alt"].copy()

df_pve01 = df_pve[df_pve["pve_level"] == 0.01].copy()

# make sure we are only using the five pcs perturbations in datasets_pve01
perturbations = [
    "anonymize",
    "shuffle_names",
    "add_features",
    "positive_leading_statement",
    "negative_leading_statement",
]
df_pve01 = df_pve01[df_pve01["perturbation"].isin(perturbations)].copy()
df_alt = df_alt[df_alt["perturbation"].isin(perturbations)].copy()


# Bootstrap mean test + overlap coefficient: alt vs pve=0.01 null
rng_pve = np.random.default_rng(42)
OVL_THRESHOLD = 0.2
MEAN_ALPHA = 0.05

datasets_pve01 = sorted(
    set(df_pve01["dataset"].unique())
)

results_pve01 = []
for ds in datasets_pve01:
    alt_scores  = df_alt.loc[df_alt["dataset"] == ds, "response"].values.astype(float)
    null_scores = df_pve01.loc[df_pve01["dataset"] == ds, "response"].values.astype(float)

    if len(alt_scores) == 0 or len(null_scores) == 0:
        continue

    alt_obs_mean, alt_p_value, _, alt_ci_95 = bootstrap_mean_test(alt_scores, mu0=50.0, rng=rng_pve)
    null_obs_mean = null_scores.mean()
    
    ovl = overlap_coefficient(alt_scores, null_scores)

    results_pve01.append({
        "dataset":  ds,
        "n_alt":    len(alt_scores),
        "n_null":   len(null_scores),
        "alt_obs_mean": alt_obs_mean,
        "alt_p_value":  alt_p_value,
        "alt_ci_lo":    alt_ci_95[0],
        "alt_ci_hi":    alt_ci_95[1],
        "null_obs_mean": null_obs_mean,
        "ovl":      ovl,
    })

results_pve01_df = pd.DataFrame(results_pve01)
results_pve01_df["alt_mean_sig"] = results_pve01_df["alt_p_value"] < MEAN_ALPHA
results_pve01_df["low_ovl"]  = results_pve01_df["ovl"] < OVL_THRESHOLD

print(results_pve01_df.to_string(index=False, float_format="%.4f"))

In [ ]:
# Scatterplot: alt vs pve=0.1 null
results_pve01_df["category"] = results_pve01_df.apply(classify, axis=1)

fig, ax = plt.subplots(figsize=(8, 6))

LABEL_FS = 20
TICK_FS  = 16
ANNOT_FS = 14
TITLE_FS = 24
LEGEND_FS = 16

texts = []
for cat in categories:
    sub = results_pve01_df[results_pve01_df["category"] == cat]
    ax.scatter(sub["ovl"], sub["alt_obs_mean"],
               color=cat_colors[cat], s=90, label=cat,
               edgecolor="white", linewidth=0.6, zorder=3)
    for _, row in sub.iterrows():
        texts.append(ax.text(
            row["ovl"], row["alt_obs_mean"], row["dataset"],
            fontsize=ANNOT_FS, fontfamily="DejaVu Sans Mono",
        ))

adjust_text(
    texts, ax=ax,
    expand=(1.3, 1.5),
    arrowprops=dict(arrowstyle="-", color="gray", lw=0.6),
)

ax.axhline(50, color="gray", linestyle="--", linewidth=0.8, zorder=1)
ax.axvline(OVL_THRESHOLD, color="gray", linestyle="--", linewidth=0.8, zorder=1)
ax.set_xlabel("Overlap Coefficient", fontsize=LABEL_FS)
ax.set_ylabel("Alt. Response Mean", fontsize=LABEL_FS)
ax.tick_params(labelsize=TICK_FS)
ax.set_xlim(-0.05, 1.05)
# ax.set_title(
#     f"Bootstrap mean test \u00d7 overlap  (threshold = {OVL_THRESHOLD},  \u03b1 = {MEAN_ALPHA})\n"
#     "alt vs pve=0.01 null",
#     fontsize=TITLE_FS,
# )
ax.legend(loc="upper right", fontsize=LEGEND_FS)
sns.despine()
plt.tight_layout()
plt.savefig("figures/alt_vs_pve001_scatter.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
# Ridgeline plot: alt vs pve=0.01 null
from scipy.stats import gaussian_kde
from matplotlib.patches import Patch

X_GRID      = np.linspace(0, 100, 500)
ROW_HEIGHT  = 1.0
RIDGE_SCALE = 0.75
NULL_COLOR  = "#008080"
ALT_COLOR   = "#FF7F50"

LABEL_FS = 20
TICK_FS  = 16
TITLE_FS = 24
LEGEND_FS = 16

n_ds = len(datasets_pve01)
fig, ax = plt.subplots(figsize=(10, n_ds * 0.7 + 1.5))

for i, ds in enumerate(reversed(datasets_pve01)):
    baseline    = i * ROW_HEIGHT
    alt_scores  = df_alt.loc[df_alt["dataset"] == ds, "response"].values.astype(float)
    null_scores = df_pve01.loc[df_pve01["dataset"] == ds, "response"].values.astype(float)

    ax.axhline(baseline, color="gray", linewidth=0.4, zorder=0)

    for scores, color in [(null_scores, NULL_COLOR), (alt_scores, ALT_COLOR)]:
        if len(scores) < 2:
            continue
        kde  = gaussian_kde(scores)
        dens = kde(X_GRID)
        dens = dens / dens.max() * RIDGE_SCALE
        ax.fill_between(X_GRID, baseline, baseline + dens,
                        color=color, alpha=0.45, linewidth=0)
        ax.plot(X_GRID, baseline + dens, color=color, linewidth=1.2)

    ax.text(-1, baseline + RIDGE_SCALE * 0.1, ds,
            ha="right", va="bottom", fontsize=TICK_FS,
            fontfamily="DejaVu Sans Mono")

ax.legend(
    handles=[Patch(color=NULL_COLOR, alpha=0.7, label="Null (PVE=0.01)"),
             Patch(color=ALT_COLOR,  alpha=0.7, label="Alt.")],
    loc="upper right", fontsize=LEGEND_FS,
)
ax.set_xlim(0, 100)
ax.set_ylim(-ROW_HEIGHT * 0.3, n_ds * ROW_HEIGHT + 0.2)
ax.set_xlabel("Response Score", fontsize=LABEL_FS)
ax.set_yticks([])
ax.tick_params(axis="x", labelsize=TICK_FS)
sns.despine(left=True)
plt.tight_layout()
plt.savefig("figures/alt_vs_pve001_ridgeline.png", dpi=600, bbox_inches="tight")
plt.show()
